# isochrone generator

Takes a given location within the **Continental United States** and outputs an **isochrone map** based on **road driving transit distance** from that location

*example_visual_output.jpeg*

Simmilar apps exist for shorter transit distances, these are generally built for intercity transit (applications like uber, walking, biking)

*simmilar_app_screenshot.jpeg*

Isochrone maps at greater distances are often created based on estimates and are not precise

*highlights_of_maps_imprecision.jpeg*

## alpha version specs
* uses h3 hexagons
* isochrone layer distances 500miles or greater
* takes input and displays output on an interactive map

## h3 core functions

- GeoToH3(lat, lng, res) - takes a point and returns what cell it's in
- H3ToGeo(h3Index) - takes a cell and returnes the geo center (opposite of GeoToH3)
- H3ToGeoBoundary(H3Index, asGeoJson) - takes a cell and returns the corners as geojson

- polyfill(coordinates, res) - convert a polygon to a set of H3 cells
- H3SetToMultiPolygon(H3Indexes) - convert a set of H3 cells to a polygon (opposite of polyfill)

- Clustering?? (when hexes are "grouped" together to draw shapes made of hexes)

## step 1 - generate a map that shows resolution 3 hexagons on a map of the CONUS

## h3 map

Uses **conus-states.json** as datasouce
- contains geometry outline of 48 CONUS states
- no AL, HI or PR
- states are unified into 1 polygon using shapely **unary_union**

Uses h3 **polygon_to_cells_expermental** with **contain="overlap"** (if any part of a hexagon is within poly it is plotted)

In [ ]:
import os
import h3
from h3 import LatLngPoly, LatLngMultiPoly
import folium
import json
from shapely.geometry import shape
from shapely.ops import unary_union

gm_api_key = os.getenv("gm_api_key")

resolution = 4

with open('datasets/conus-states.json', 'r') as f:
    # Parsing the JSON file into a Python dictionary
    state_geo = json.load(f)

m = folium.Map(location=(40, -100), zoom_start=5)

geoms = [shape(f["geometry"]) for f in state_geo["features"]]
conus = unary_union(geoms) # conus is shaprely

def convert_shapely_to_h3(geom):
    if geom.geom_type == "Polygon":
        exterior = [(lat, lon) for lon, lat in geom.exterior.coords]
        holes = [
            [(lat, lon) for lon, lat in ring.coords]
            for ring in geom.interiors
        ]
        return LatLngPoly(exterior, holes)

    elif geom.geom_type == "MultiPolygon":
        return LatLngMultiPoly(*(convert_shapely_to_h3(p) for p in geom.geoms))

hexes = h3.polygon_to_cells_experimental(convert_shapely_to_h3(conus), res=resolution, contain="overlap")

for cell in hexes:
    outline = h3.cell_to_boundary(cell)
    folium.Polygon(locations=outline, weight=1, fill=False, color="black").add_to(m)

def add_geom(g):
    if g.geom_type == "Polygon":
        folium.Polygon(
            locations=[(lat, lon) for lon, lat in g.exterior.coords],
            weight=2,
            #color="yellow",
            fill=False
        ).add_to(m)
    elif g.geom_type == "MultiPolygon":
        for p in g.geoms:
            add_geom(p)
 
add_geom(conus)

# center point of each cell
centers = {}
for cell in hexes:
    center = h3.cell_to_latlng(cell)
    folium.CircleMarker(
    location= center,
    radius=2,
    ).add_to(m)
    centers.update({cell: center})
m

## Full Origion/destination vs Neighbor traversal

### Option 1 Origin/destination pairing - large table contains transit distances for all cells to all other cells
With a simple o/d pairing table, data requirements increase exponentially as cell resolution increases (might try - Identify only those hexagons whos neighbors fall on the next isochrone layer and "increase resolution" for those cells only (this is only a solution if the resulting map is step based, this would not support a gradient distance map))

$$T(r) \sim 1.6 \cdot 49^{r}$$

### Option 2 Neighbor traversal - mileage found by traversing from one cell to neighboring cells repeatedly
this method would require high resolution to avoid "side road center snapping issue" ie what happens when road snapped hexagon centers go "out of the way" (imagine row of hexes following a straight highway with lots of nearby sideroads, we want to ensure if many snapped centers fall on side roads this does not result in some calculated transit distance including the distance of taking each exit and getting back on the highway)


| Resolution | Hex Edge (km) | Hexagons | Values in O/D Table | Values in Neighbor Table | 
|------------|---------------|----------|---------------------|--------------------------|
| 2          | 182.512       | 122      | 7,381               | 366                      |
| 3          | 68.989        | 701      | 245,350             | 2,103                    |
| 4          | 26.071        | 4,533    | 10,271,778          | 13,599                   |
| 5          | 9.854         | 30,712   | 471,598,116         | 92,136                   |
| 6          | 3.724         | 205,810  | 23.8 B              | 617,711                  |
| 7          | 1.406         | 1.38 M   | 1.31 T              | 4,137,520                |
| 8          | 0.531         | 9.66 M   | 78.6 T              | 9,660,000                |
| 9          | 0.200         | 67.2 M   | -                   | 67,200,000               |

*approximates & averages

## Google Maps Nearest Road API

Limitations:
- This API is targeted for "jittery GPS" and helps to snap to roads within ~15meters
- To ensure a hexagon is searched entirely, resolution of 13 or higher would be necessary (average hexagon size at res13=0.000043870 km2)
- targeting resolution of 3-6 for this application

Possible solutions:
(possible looping search solution using fractal type search)

- For *this* application greater snap distances are needed

In [ ]:
import requests
import os
gm_api_key = os.getenv("gm_api_key")


# Point to snap (latitude,longitude)
lat = 37.516956 
lng = -103.330191

url = "https://roads.googleapis.com/v1/nearestRoads"


params = {
    "points": f"{lat},{lng}",
    "key": gm_api_key
}

print(url, params)

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

# Print full response
print(data)

# Extract nearest road info (if available)
snapped_points = data.get("snappedPoints", [])

if snapped_points:
    nearest = snapped_points[0]
    location = nearest["location"]
    place_id = nearest["placeId"]

    print("\nNearest road result:")
    print(f"Snapped latitude:  {location['latitude']}")
    print(f"Snapped longitude: {location['longitude']}")
    print(f"Place ID:          {place_id}")
else:
    print("No nearby road found.")
